# Random-Forest Baseline v3

Dieses Notebook bewertet `v3` mit der zusammengelegten Klasse `Kein Schlag / Schlaeger drehen` und vergleicht die Ergebnisse direkt mit `v1` und `v2`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "random_forest_baseline_v3").exists():
            return candidate
    raise FileNotFoundError("Repo root mit src/random_forest_baseline_v3 wurde nicht gefunden.")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.random_forest_baseline.config import BaselineConfig
from src.random_forest_baseline.data import prepare_dataset
from src.random_forest_baseline.modeling import evaluate_grouped_dataset, save_confusion_matrix_plot

config = BaselineConfig.from_json(repo_root / "baseline_models/random_forest_v3/config.json")
config


In [ ]:
search_results_path = repo_root / "baseline_models/random_forest_v3/output/search_results.csv"
search_summary_path = repo_root / "baseline_models/random_forest_v3/output/search_summary.json"

if search_results_path.exists():
    display(pd.read_csv(search_results_path).head(10))

if search_summary_path.exists():
    search_summary = json.loads(search_summary_path.read_text())
    display(pd.DataFrame([search_summary["best_metrics"]]))
    display(pd.DataFrame([search_summary["best_params"]]))


In [ ]:
config_v2 = BaselineConfig.from_json(repo_root / "baseline_models/random_forest_v2/config.json")
dataset_v2 = prepare_dataset(config_v2)
dataset_v3 = prepare_dataset(config)

label_compare = pd.concat(
    [
        dataset_v2.label_summary.assign(model="v2"),
        dataset_v3.label_summary.assign(model="v3"),
    ],
    ignore_index=True,
)
display(label_compare)
display(dataset_v3.metadata.head())
print(f"v3 Events: {len(dataset_v3.labels)} | Sessions: {dataset_v3.groups.nunique()} | Features: {dataset_v3.features.shape[1]}")


In [ ]:
result = evaluate_grouped_dataset(dataset_v3, config)
display(pd.DataFrame([result.overall_metrics]))
display(result.classification_report)
display(result.fold_metrics)


In [ ]:
figure = save_confusion_matrix_plot(result.confusion_matrix)
figure


In [ ]:
metric_paths = {
    "v1": repo_root / "baseline_models/random_forest/output/metrics.json",
    "v2": repo_root / "baseline_models/random_forest_v2/output/metrics.json",
    "v3": repo_root / "baseline_models/random_forest_v3/output/metrics.json",
}

rows = []
for model_name, path in metric_paths.items():
    payload = json.loads(path.read_text())
    rows.append({"model": model_name, **payload["overall_metrics"]})

comparison = pd.DataFrame(rows).set_index("model").loc[["v1", "v2", "v3"]]
comparison["accuracy_vs_v2"] = comparison["accuracy"] - comparison.loc["v2", "accuracy"]
comparison["macro_f1_vs_v2"] = comparison["macro_f1"] - comparison.loc["v2", "macro_f1"]
comparison["balanced_accuracy_vs_v2"] = comparison["balanced_accuracy"] - comparison.loc["v2", "balanced_accuracy"]
display(comparison)


In [ ]:
feature_importances_path = repo_root / "baseline_models/random_forest_v3/output/feature_importances.csv"
if feature_importances_path.exists():
    display(pd.read_csv(feature_importances_path).head(20))


## Anpassung

Wenn ihr die gemergte Klasse wieder aufteilen oder andere Fenster-/Forest-Parameter testen wollt, passt zuerst `baseline_models/random_forest_v3/config.json` an und fuehrt die Zellen erneut aus.